## Environment Setup

In [12]:
import sys
from pathlib import Path
import requests

print(f"Python Version: {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")
print(f"Environment: {sys.executable}")

current_dir = Path.cwd()
if current_dir.name == "tests":
    project_root = current_dir.parent
else:
    project_root = current_dir

print(f"Project Root: {project_root}")

if project_root and (project_root / "src").exists():
    sys.path.insert(0, str(project_root))
else:
    print("Project root not found or src directory missing")
    sys.exit(1)


Python Version: 3.12.11
Environment: /Users/xieqiqi/Learning/LLM/my_first_Rag_project/.venv/bin/python3
Project Root: /Users/xieqiqi/Learning/LLM/my_first_Rag_project


## 1. Infrastructure Verification

In [13]:
# Test opensearch health check
import requests

try:
    response = requests.get("http://localhost:9200/_cluster/health", timeout=5)
    if response.status_code == 200:
        data = response.json()
        print("✓ OpenSearch is responding")
        print(f"Status: {data.get('status', 'unknown')}")
    else:
        print(f"⚠ API returned status: {response.status_code}")
except requests.exceptions.ConnectionError as e:
    print(f"⚠ API not responding: {e}")
except Exception as e:
    print(f"⚠ API test error: {e}") 


✓ OpenSearch is responding
Status: yellow


## 2. Opensearch Client Setup

In [14]:
from src.services.opensearch.factory import make_opensearch_client
from opensearchpy import OpenSearch
print("Opensearch client setup")
print("="*40)

opensearch_client = make_opensearch_client()
opensearch_client.host = "http://localhost:9200"
opensearch_client.client = OpenSearch(
    hosts=["http://localhost:9200"],
    http_compress=True,
    use_ssl=False,
    verify_certs=False,
    ssl_assert_hostname=False,
    ssl_show_warn=False
)

print(f"Client configured with host: {opensearch_client.host}")
print(f"Index name: {opensearch_client.index_name}")

is_healthy = opensearch_client.health_check()
if is_healthy:
    print("✓ OpenSearch health check: PASSED")
else:
    print("⚠ OpenSearch health check: FAILED")



Opensearch client setup
Client configured with host: http://localhost:9200
Index name: arxiv-papers-chunks
✓ OpenSearch health check: PASSED


## Create Index

In [15]:
# Create index if it doesn't exist
print("Index creation")
print("="*40)
try:
    index_exists = opensearch_client.client.indices.exists(index=opensearch_client.index_name)
    if not index_exists:
        print(f"Creating new index: {opensearch_client.index_name}")
        success = opensearch_client.client.indices.create(index=opensearch_client.index_name)
        if success:
            print("Index created successfully")
        else:
            print("Index creation failed")
    else:
        print("Index already exists")
        stats = opensearch_client.get_index_stats()
        if stats and 'error' not in stats:
            print(f"\nCurrent Statistics:")
            print(f"   Documents: {stats.get('document_count', 0)}")
            print(f"   Size: {stats.get('size_in_bytes', 0):,} bytes")
except Exception as e:
    print(f"Failed to create index: {e}")

Index creation
Index already exists

Current Statistics:
   Documents: 0
   Size: 208 bytes


## 3. Data Pipeline Test

In [16]:
# Verify Data Pipeline Results
print("Verifying data pipeline results")
print("="*40)

stats = opensearch_client.get_index_stats()

if stats and 'error' not in stats:
    doc_count = stats.get('document_count', 0)

    if doc_count > 0:
        print(f"Success! Found {doc_count} documents in OpenSearch")
    
        sample = opensearch_client.search_paper("*", size=3)
        if sample.get("hits"):
            print(f"\nSample papers:")
            for i, paper in enumerate(sample.get("hits")):
                title = paper.get("title", "Unknown")[:60]
                print(f"Paper {i}: {title}")

        else:
            print("No documents found in OpenSearch")
            print("\nPlease run the Airflow DAG first")
else:
    print("✗ Could not retrieve index stats")

Verifying data pipeline results


## 4. Simple BM25 Search

In [ ]:
print("Simple BM25 search")
print("="*40)

search_term = "learning"

print("Searching for:'{search_term}'\n")
results = opensearch_client.search_papers(
    query=search_term,
    size=5
)

if results.get('hits'):
    print(f"Found {results.get('total', 0)} total matches\n")
    
    for i, paper in enumerate(results['hits'], 1):
        print(f"{i}. {paper.get('title', 'Unknown')[:70]}...")
        print(f"   Score: {paper.get('score', 0):.2f}")
        print(f"   arXiv ID: {paper.get('arxiv_id', 'N/A')}\n")
else:
    print("No results found. Try searching for:")
    print("  • 'neural', 'model', 'algorithm'")
    print("  • Use '*' to see all papers")
